# LLM Wiki — A Self-Generating Wiki with Graph Visualization

This notebook builds a small **encyclopedia about LLMs, authored entirely by an LLM**. It runs **entirely locally**, except for the page-generation and answer calls, which use **Groq's `llama-3.3-70b-versatile`**.

Starting from a single root topic (`"Large Language Models"`), the notebook asks Groq to write a wiki article for the topic *and* list related topics ("See also" links) — the same way a real wiki grows. It then **crawls** those links breadth-first, generating new pages up to a page budget, exactly like following hyperlinks between Wikipedia articles.

Pipeline:
1. **Seed** the crawl with a root topic
2. **Generate** wiki pages with Groq (structured JSON: title, article body, related-topic links) and **crawl** the link graph breadth-first up to a page budget
3. **Persist** each page as a Markdown file under `wiki/`
4. **Build** a directed link graph (`networkx`) — nodes are pages, edges are `page → linked page`
5. **Visualize** the wiki as an interactive, force-directed graph (inline Plotly + a standalone draggable [pyvis](https://pyvis.readthedocs.io/) HTML file)
6. **Chunk + embed** each page's text locally and store it in a local vector DB ([ChromaDB](https://www.trychroma.com/)) for retrieval
7. **Ask a question** → retrieve the most relevant chunks, highlight their source pages in the wiki graph
8. **Compile an answer** by sending the question + retrieved context to Groq's Llama 3.3 70B model

Only Groq is used as an external API (for page generation + answer generation) — crawling, chunking, embeddings, the vector DB, and both graph visualizations run locally.

## 0. Setup

This notebook expects to run under the `.venv` virtual environment in this project folder — select **Python (.venv)** as the kernel (top-right in VS Code / Jupyter) before running the cells below.

Dependencies are listed in `requirements.txt`; the cell below installs them into `.venv`. API keys are loaded from a local `.env` file via `python-dotenv` — copy `.env.example` to `.env` and fill in your Groq key (free tier at https://console.groq.com/keys):

```
GROQ_API_KEY=your-key-here
```

`.env` is never read by anything outside this notebook/process, so it keeps the key out of the notebook source itself.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os
import re
import json
import time
from collections import deque
from pathlib import Path

import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from dotenv import load_dotenv

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from pyvis.network import Network

load_dotenv()

WIKI_DIR = Path("wiki")                   # generated Markdown pages
WIKI_GRAPH_DIR = Path("wiki_graph_db")     # persisted graph + rendered HTML views
MODEL_CACHE_DIR = Path(".model_cache")     # local cache so the embedding model isn't re-downloaded each run

ROOT_TOPIC = "Large Language Models"
MAX_PAGES = 12            # page budget for the crawl (keeps the demo fast & inside free-tier rate limits)
MAX_LINKS_PER_PAGE = 5    # cap on "See also" links Groq returns per page, to keep the graph readable

CHUNK_SIZE = 500          # characters per chunk
CHUNK_OVERLAP = 80        # overlap between consecutive chunks
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.3-70b-versatile"
TOP_K = 4                 # number of chunks to retrieve per question
HOPS = 1                  # how many link hops to expand around the retrieved pages, for the highlighted graph view

WIKI_DIR.mkdir(exist_ok=True)
WIKI_GRAPH_DIR.mkdir(exist_ok=True)


## 1. Generate the wiki with Groq (breadth-first crawl)

For each topic in the crawl queue, ask Groq for a JSON object with the page's title, article body, and a short list of related topics to link to. New, not-yet-seen topics are pushed onto the queue — the same breadth-first process a reader follows when clicking through wiki links — until `MAX_PAGES` pages have been generated.

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY not found — copy .env.example to .env and add your key."
groq_client = Groq(api_key=groq_api_key)

WIKI_SYSTEM_PROMPT = (
    "You are a wiki author writing a small, focused encyclopedia about large language models "
    "and related AI/NLP concepts. Respond only with strict JSON, no prose."
)

def normalize(title):
    return re.sub(r"\s+", " ", title.strip().lower())

def slugify(title):
    return re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-") or "page"

def generate_wiki_page(topic):
    prompt = f"""Write a wiki article about: "{topic}"

Respond as JSON:
{{
  "title": "canonical article title",
  "content": "3-5 paragraphs explaining the topic, written encyclopedia-style",
  "links": ["up to {MAX_LINKS_PER_PAGE} closely related topics worth their own wiki article"]
}}"""
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": WIKI_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

pages = {}          # canonical title -> {"content": str, "links": [str, ...]}
visited = set()     # normalized titles already crawled (or attempted)
queue = deque([ROOT_TOPIC])

while queue and len(pages) < MAX_PAGES:
    topic = queue.popleft()
    key = normalize(topic)
    if key in visited:
        continue
    visited.add(key)

    try:
        data = generate_wiki_page(topic)
    except (json.JSONDecodeError, Exception) as e:
        print(f"  [skip] {topic!r}: {e}")
        continue

    title = str(data.get("title") or topic).strip()
    content = str(data.get("content", "")).strip()
    links = [str(l).strip() for l in data.get("links", [])[:MAX_LINKS_PER_PAGE] if str(l).strip()]

    pages[title] = {"content": content, "links": links}
    print(f"  [{len(pages)}/{MAX_PAGES}] generated {title!r} — {len(links)} link(s)")

    for link in links:
        if normalize(link) not in visited:
            queue.append(link)

    time.sleep(0.3)  # gentle pacing against Groq's free-tier rate limits

print(f"\nCrawled {len(pages)} pages, {len(queue)} link(s) left unexplored at the page budget")


## 2. Persist the wiki as Markdown

Each generated page is written to `wiki/<slug>.md` with a "See also" section listing its outgoing links — a plain-text wiki you can browse directly on disk.

In [ ]:
slug_by_title = {title: slugify(title) for title in pages}

for title, data in pages.items():
    see_also = "\n".join(f"- {link}" for link in data["links"]) or "- (none)"
    page_md = f"# {title}\n\n{data['content']}\n\n## See also\n{see_also}\n"
    (WIKI_DIR / f"{slug_by_title[title]}.md").write_text(page_md, encoding="utf-8")

print(f"Wrote {len(pages)} Markdown pages to {WIKI_DIR}/")
pd.DataFrame([{"title": t, "slug": slug_by_title[t], "links": len(d["links"])} for t, d in pages.items()])


## 3. Build the link graph

Nodes are wiki pages; a directed edge `A → B` means page `A` links to page `B`. Linked topics that were discovered but never crawled (because the page budget ran out) still appear as **stub nodes** — they show where the wiki would keep growing.

In [ ]:
norm_to_title = {normalize(t): t for t in pages}

def resolve_link(link):
    """Map a link string to an already-crawled page's canonical title, if there is one."""
    return norm_to_title.get(normalize(link), link)

G = nx.DiGraph()
for title in pages:
    G.add_node(title, crawled=True)

for title, data in pages.items():
    for link in data["links"]:
        target = resolve_link(link)
        if target not in G:
            G.add_node(target, crawled=False)
        G.add_edge(title, target)

graphml_path = WIKI_GRAPH_DIR / "wiki_graph.graphml"
nx.write_graphml(G, graphml_path)

n_crawled = sum(1 for _, d in G.nodes(data=True) if d.get("crawled"))
n_stub = G.number_of_nodes() - n_crawled
print(f"Graph: {G.number_of_nodes()} nodes ({n_crawled} crawled, {n_stub} stub), {G.number_of_edges()} links")
print(f"Persisted to {graphml_path}")


## 4. Visualize the wiki graph

Two views are produced:

- An **inline Plotly view** (below) — node labels are always visible; hover over a node for its status, hover over an edge for the link direction. Renders directly in the notebook output, scroll to zoom, drag to pan.
- A **standalone [pyvis](https://pyvis.readthedocs.io/) HTML file**, saved to `wiki_graph_db/` — a draggable, physics-based force layout. VS Code's notebook output sandbox doesn't run the `<script>` this needs to draw itself, so it can't be embedded inline — open the saved file directly in a browser tab for the fully interactive version.

Crawled pages are blue, stub (linked-but-not-generated) topics are grey, and (later) retrieved pages are highlighted in red.

In [ ]:
def build_plotly_graph(graph, highlight_nodes=None, title="Wiki graph", height=650, seed=42):
    highlight_nodes = highlight_nodes or set()
    pos = nx.spring_layout(graph, seed=seed)

    edge_x, edge_y = [], []
    for u, v in graph.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1, color="#666"),
        hoverinfo="none", showlegend=False,
    )

    node_x, node_y, node_text, node_hover, node_color = [], [], [], [], []
    for node, data in graph.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
        status = "crawled page" if data.get("crawled") else "stub (not yet generated)"
        node_hover.append(f"{node}<br>{status}")
        if node in highlight_nodes:
            node_color.append("#e74c3c")
        elif data.get("crawled"):
            node_color.append("#3498db")
        else:
            node_color.append("#95a5a6")

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=node_text, textposition="top center",
        textfont=dict(color="#e0e0e0", size=11),
        hoverinfo="text", hovertext=node_hover,
        marker=dict(size=16, color=node_color, line=dict(width=1, color="#1e1e1e")),
        showlegend=False,
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        plot_bgcolor="#1e1e1e", paper_bgcolor="#1e1e1e",
        font=dict(color="#e0e0e0"),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        margin=dict(l=0, r=0, t=40, b=0),
        height=height,
    )
    return fig

def export_pyvis_html(graph, path, highlight_nodes=None, height="650px"):
    highlight_nodes = highlight_nodes or set()
    net = Network(height=height, width="100%", directed=True, notebook=True,
                   cdn_resources="in_line", bgcolor="#1e1e1e", font_color="#e0e0e0")
    net.barnes_hut()

    for node, data in graph.nodes(data=True):
        status = "crawled page" if data.get("crawled") else "stub (not yet generated)"
        if node in highlight_nodes:
            color = "#e74c3c"
        elif data.get("crawled"):
            color = "#3498db"
        else:
            color = "#95a5a6"
        net.add_node(node, label=node, title=f"{node}<br>{status}", color=color)

    for u, v in graph.edges():
        net.add_edge(u, v, arrows="to")

    page = net.generate_html(notebook=True)
    out_path = Path(path).resolve()
    out_path.write_text(page, encoding="utf-8")
    return out_path

full_graph_path = export_pyvis_html(G, WIKI_GRAPH_DIR / "wiki_graph_full.html")
print(f"Draggable, physics-based view saved to: {full_graph_path}\n(open it directly in a browser tab)")

build_plotly_graph(G, title="Full wiki graph").show()


## 5. Chunk and embed the wiki pages

Same fixed-size sliding-window splitter and local sentence-transformer embedding model as `rag_demo.ipynb`. Only crawled pages have article text to chunk — stub nodes exist purely as graph placeholders.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

records = []  # each: {id, text, source}
for title, data in pages.items():
    for i, chunk in enumerate(chunk_text(data["content"])):
        records.append({"id": f"{title}::{i}", "text": chunk, "source": title})

print(f"Created {len(records)} chunks from {len(pages)} pages")

embedder = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=str(MODEL_CACHE_DIR))
texts = [r["text"] for r in records]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)

for r, e in zip(records, embeddings):
    r["embedding"] = e

chroma_client = chromadb.PersistentClient(path="./chroma_db_wiki")
collection = chroma_client.get_or_create_collection(
    name="wiki_chunks",
    metadata={"hnsw:space": "cosine"},
)

# Reset collection contents so re-running the notebook doesn't duplicate chunks
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)

collection.add(
    ids=[r["id"] for r in records],
    embeddings=[r["embedding"].tolist() for r in records],
    documents=[r["text"] for r in records],
    metadatas=[{"source": r["source"]} for r in records],
)

print(f"Stored {collection.count()} chunks in the vector DB")


## 6. Ask a question — retrieve chunks and highlight their source pages

Embed the question, run a similarity search against the vector DB, then highlight the retrieved pages (and their direct links, out to `HOPS` hops) in the wiki graph so you can see where the answer is coming from.

In [ ]:
def retrieve(question, k=TOP_K):
    q_embedding = embedder.encode([question], normalize_embeddings=True)[0]
    results = collection.query(query_embeddings=[q_embedding.tolist()], n_results=k)
    hits = []
    for text, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        hits.append({"text": text, "source": meta["source"], "distance": dist})
    return hits, q_embedding

def neighborhood_subgraph(seed_pages, hops=HOPS):
    nodes_in_subgraph = set(seed_pages)
    frontier = set(seed_pages)
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            if node in G:
                next_frontier.update(G.predecessors(node))
                next_frontier.update(G.successors(node))
        nodes_in_subgraph.update(next_frontier)
        frontier = next_frontier
    return G.subgraph(nodes_in_subgraph).copy()

question = "How does attention work in a transformer, and how does it relate to fine-tuning?"
hits, q_embedding = retrieve(question)

for h in hits:
    print(f"[{h['source']}] (distance={h['distance']:.3f})\n{h['text']}\n")

seed_pages = {h["source"] for h in hits}
subgraph = neighborhood_subgraph(seed_pages)

subgraph_path = export_pyvis_html(subgraph, WIKI_GRAPH_DIR / "wiki_graph_subgraph.html", highlight_nodes=seed_pages, height="500px")
print(f"Draggable, physics-based view saved to: {subgraph_path}\n(open it directly in a browser tab)")

build_plotly_graph(subgraph, highlight_nodes=seed_pages, title=f'Retrieved pages for: "{question}"', height=500).show()


## 7. Compile an answer with Groq (`llama-3.3-70b-versatile`)

The retrieved chunks are stitched into a context block and sent to the LLM along with the question — the same pattern as `rag_demo.ipynb`.

In [ ]:
def compile_answer(question, hits):
    context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits)
    prompt = f"""Answer the question using only the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that answers strictly from the provided context and cites the source page(s) you used."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

answer = compile_answer(question, hits)
print(answer)


## 8. End-to-end `ask()` helper

Combine retrieval + graph highlighting + generation into a single function for interactive use.

In [ ]:
def ask(question, k=TOP_K, hops=HOPS, verbose=True):
    hits, _ = retrieve(question, k=k)
    seed_pages = {h["source"] for h in hits}
    subgraph = neighborhood_subgraph(seed_pages, hops=hops)
    answer = compile_answer(question, hits)

    build_plotly_graph(subgraph, highlight_nodes=seed_pages, title=f'Retrieved pages for: "{question}"', height=450).show()

    if verbose:
        print("Retrieved chunks:")
        for h in hits:
            print(f"  - [{h['source']}] {h['text'][:100]}...")
        print("\nAnswer:\n" + answer)
    return answer

# Try it yourself:
ask("What are the main differences between fine-tuning and RAG?")


## Next steps

- Raise `MAX_PAGES` to let the crawl grow the wiki further (slower, more Groq calls — watch your rate limits)
- Change `ROOT_TOPIC` to seed a wiki about a completely different domain
- Improve link resolution — right now merging is exact normalized-string match, so near-duplicate topic names ("RLHF" vs "Reinforcement Learning from Human Feedback") become separate nodes; try fuzzy matching or an LLM-based dedup pass
- Try `HOPS = 2` for broader highlighted context, or prune low-connectivity stub nodes before visualizing
- Reload the wiki from `wiki/*.md` and the graph from `wiki_graph_db/wiki_graph.graphml` on startup instead of re-crawling every run
- Swap `nx.spring_layout(...)` for `nx.kamada_kawai_layout(...)` for a different inline layout, or tune `net.set_options(...)` (pyvis) for the browser-viewed file